# Minestone 2: Heuristic Mining

- Due to runtime constraints, a random sample of 5000 cases was taken from the clean log. The noised and recovered logs were then regenerated by applying the pollution and cleaning pipelines developed in Milestone 1 to this sample. This ensured that all three logs were derived from the same set of cases and maintained a noise proportion comparable to that of the original logs

In [1]:
import pm4py
import pandas as pd
from pathlib import Path
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner

from pm4py.visualization.heuristics_net import visualizer as hn_vis
from pm4py.visualization.petri_net import visualizer as pn_vis

from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator

In [2]:
DATA_DIR = Path.cwd().parent / "data"
SAMPLE_DIR = DATA_DIR / "samples"
OUTPUT_DIR = DATA_DIR / "milestone2"

In [3]:
def load_log(filename: str):
    path = DATA_DIR / filename
    log = pm4py.read_xes(str(path))
    df = pm4py.convert_to_dataframe(log)
    print(f"Loaded {filename}: {len(df)} events, "
          f"{df['case:concept:name'].nunique()} cases")
    return log, df


def compute_frequency_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute |a >L b|: how many times activity a is directly
    followed by activity b across all cases.
    """
    df = df.sort_values(["case:concept:name", "time:timestamp"])

    rows = []
    for case_id, group in df.groupby("case:concept:name"):
        activities = group["concept:name"].tolist()
        for i in range(len(activities) - 1):
            rows.append({
                "from": activities[i],
                "to":   activities[i + 1]
            })

    freq_df = pd.DataFrame(rows)
    freq_table = freq_df.groupby(["from", "to"]).size().reset_index(name="frequency")
    return freq_table


def compute_dependency(freq_table: pd.DataFrame, a: str, b: str) -> dict:
    """
    Compute dependency value dep(a→b) manually from frequency counts.
    Also returns the raw counts for manual verification.

    Formula:
        a ≠ b: dep(a→b) = (|a>b| - |b>a|) / (|a>b| + |b>a| + 1)
        a = b: dep(a→a) = |a>a| / (|a>a| + 1)
    """
    def get_freq(x, y):
        row = freq_table[(freq_table["from"] == x) & (freq_table["to"] == y)]
        return int(row["frequency"].values[0]) if not row.empty else 0

    ab = get_freq(a, b)
    ba = get_freq(b, a)

    if a == b:
        dep = ab / (ab + 1)
    else:
        dep = (ab - ba) / (ab + ba + 1)

    return {
        "a": a, "b": b,
        f"|{a} > {b}|": ab,
        f"|{b} > {a}|": ba,
        "dependency": round(dep, 4)
    }

In [4]:
log_clean,     df_clean     = load_log(SAMPLE_DIR / "clean_sample.xes.gz")
log_noised,    df_noised    = load_log(SAMPLE_DIR / "noised_sample.xes.gz")
log_recovered, df_recovered = load_log(SAMPLE_DIR / "recovered_sample.xes.gz")

c:\Users\thien\Downloads\Process Mining\ProM-Assignment-Group-C\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
c:\Users\thien\Downloads\Process Mining\ProM-Assignment-Group-C\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 5000/5000 [00:26<00:00, 189.78it/s]


Loaded c:\Users\thien\Downloads\Process Mining\ProM-Assignment-Group-C\data\samples\clean_sample.xes.gz: 190334 events, 5000 cases


parsing log, completed traces :: 100%|██████████| 5000/5000 [00:33<00:00, 149.23it/s]


Loaded c:\Users\thien\Downloads\Process Mining\ProM-Assignment-Group-C\data\samples\noised_sample.xes.gz: 151673 events, 5000 cases


parsing log, completed traces :: 100%|██████████| 5000/5000 [00:57<00:00, 87.57it/s] 


Loaded c:\Users\thien\Downloads\Process Mining\ProM-Assignment-Group-C\data\samples\recovered_sample.xes.gz: 190337 events, 5000 cases


In [5]:
logs_dict = {
    "clean": log_clean,
    "recovered": log_recovered,
    "noised": log_noised
}

df_dict = {
    "clean": df_clean,
    "recovered": df_recovered,
    "noised": df_noised
}

## Task 1: Compute the dependency graph
- Nodes are activities, edges are causal arcs that pass both your frequency and dependency thresholds, annotated with frequency(dependency)
- For at least one activity pair, show the dependency calculation manually (starting from the frequency counts) and verify against the value reported by the tool.

In [6]:
act1 = "O_Created"
act2 = "O_Create Offer"

### Manual Calculation

In [7]:
freq_tables = {}
results = {}

for name, df in df_dict.items():
    freq_tables[name] = compute_frequency_table(df)
    results[name] = compute_dependency(freq_tables[name], act1, act2)

df = pd.DataFrame(results)
print(df)

                                       clean       recovered          noised
a                                  O_Created       O_Created       O_Created
b                             O_Create Offer  O_Create Offer  O_Create Offer
|O_Created > O_Create Offer|             617             617             553
|O_Create Offer > O_Created|            6869            6869            6191
dependency                            -0.835          -0.835         -0.8359


### Built-in function

In [8]:
rows = []
nets = {}

for name, log in logs_dict.items():
    heu_net = heuristics_miner.apply_heu(log)
    nets[name] = heu_net
    gviz = hn_vis.apply(heu_net)
    gviz.format = "svg"
    gviz.render(
        filename=f"{OUTPUT_DIR}/hm_dependency_graph_{name}",
        cleanup=True
    )

for name, net in nets.items():
    dep = net.dependency_matrix[act1][act2]
    fwd = net.dfg.get((act1, act2), 0)
    bwd = net.dfg.get((act2, act1), 0)

    rows.append({
        "log": name,
        "from": act1,
        "to": act2,
        "forward": fwd,
        "backward": bwd,
        "dependency": round(dep, 4)
    })

pd.DataFrame(rows)

,log,from,to,forward,backward,dependency
0,clean,O_Created,O_Create Offer,617,6869,-0.8350
1,recovered,O_Created,O_Create Offer,617,6869,-0.8350
2,noised,O_Created,O_Create Offer,553,6191,-0.8359


**Justification:**
- `dependency_threshold = 0.7` keeps only arcs with reasonably strong directional evidence.
- `min_act_count = 400` and `min_dfg_occurrences = 400`: These values were chosen based on the dependency graphs of the clean, noised and recovered logs. While the threshold may be high enough to remove some meaningful low-frequency activities and relations from the clean and recovered logs, this is intentionally chosen for some of the noise in noised log introduced in Milestone 1 occurs with relatively high frequency

In [9]:
PARAMS = {
    "dependency_threshold": 0.7,
    "min_act_count": 400,
    "min_dfg_occurrences": 400,
}

### Task 2: Production of Heuristic Net (omit)
- Run Heuristics Miner with identical parameters across the three logs to produce three Heuristic Nets 

In [ ]:
# nets_filtered = {}
# for name, log in logs_dict.items():
#     heu_net = heuristics_miner.apply_heu(log, parameters=PARAMS)
#     nets_filtered[name] = heu_net
#     gviz = hn_vis.apply(heu_net)
#     gviz.format = "svg"
#     gviz.render(
#         filename=f"{OUTPUT_DIR}/hm_heuristic_net_{name}",
#         cleanup=True
#     )

### Task 3: Convert to a Petri net and compute the four quality dimensions

In [11]:
# Stats for logs applying heuristics miner with filtering parameters

stats = {}
for name, log in logs_dict.items():
    net, im, fm = heuristics_miner.apply(log, parameters=PARAMS)
    gviz = pn_vis.apply(net, im, fm)
    gviz.format = "svg"
    gviz.render(
        filename=f"{OUTPUT_DIR}/hm_petri_net_{name}",
        cleanup=True
    )
    
    fitness = replay_fitness.apply(log, net, im, fm, variant=replay_fitness.Variants.TOKEN_BASED)
    precision = precision_evaluator.apply(log, net, im, fm, variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN)
    generalization = generalization_evaluator.apply(log, net, im, fm)
    simplicity = simplicity_evaluator.apply(net)
    
    stats[name] = {
        "fitness": round(fitness["log_fitness"], 4),
        "precision": round(precision, 4),
        "generalization": round(generalization, 4),
        "simplicity": round(simplicity, 4)
    }
pd.DataFrame(stats)

replaying log with TBR, completed traces :: 100%|██████████| 4745/4745 [00:54<00:00, 86.37it/s] 


,clean,recovered,noised
fitness,0.9413,0.9031,0.9231
precision,0.7671,0.5446,0.5069
generalization,0.9749,0.9477,0.9360
simplicity,0.5766,0.6209,0.5327


In [14]:
# Stats for logs applying heuristics miner without filtering parameters

stats_unfiltered = {}
for name, log in logs_dict.items():
    net, im, fm = heuristics_miner.apply(log)
   
    fitness = replay_fitness.apply(log, net, im, fm, variant=replay_fitness.Variants.TOKEN_BASED)
    precision = precision_evaluator.apply(log, net, im, fm, variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN)
    generalization = generalization_evaluator.apply(log, net, im, fm)
    simplicity = simplicity_evaluator.apply(net)
    
    stats_unfiltered[name] = {
        "fitness": round(fitness["log_fitness"], 4),
        "precision": round(precision, 4),
        "generalization": round(generalization, 4),
        "simplicity": round(simplicity, 4)
    }
pd.DataFrame(stats_unfiltered)

replaying log with TBR, completed traces :: 100%|██████████| 4745/4745 [03:27<00:00, 22.86it/s]


,clean,recovered,noised
fitness,0.9537,0.9041,0.9137
precision,0.7466,0.5446,0.3914
generalization,0.9193,0.8839,0.4980
simplicity,0.5138,0.5502,0.4151


In [16]:
pd.DataFrame(stats_unfiltered).T

,fitness,precision,generalization,simplicity
clean,0.9537,0.7466,0.9193,0.5138
recovered,0.9041,0.5446,0.8839,0.5502
noised,0.9137,0.3914,0.4980,0.4151


### Notes:
- Applying frequency and dependency filters significantly improves the quality of a process discovery model when dealing with noised logs. This highlights the primary strength of the heuristic mining method in handling noises.

- **Analysis for logs with filtering parameters**:
    - **Clean Log:** Clean log achieves the best overall quality.
    - **Noised Log:** Noised log has the lowest precision and generalization. Noise changes activity frequencies and dependency values, causing incorrect arcs to survive the thresholds and making the model allow more behavior than intended.
    - **Recovered Log**: The recovered log performs between the clean and noised logs. Some noisy relations were removed, improving the model compared to the noised log. The recovered model also has the highest simplicity score (0.6209), suggesting that several noisy or infrequent relations were removed. However, some valid behavior was also lost, which explains the lower fitness compared to the clean log.

- **Effect of M1 Imperfections**: The noise introduced in Milestone 1 created additional directly-follows relations and changed dependency values. As a result:
    - Some valid arcs fell below the threshold and were removed.
    - Some noisy arcs passed the thresholds and were included.
    - The discovered process structure changed, leading to lower precision and generalization. The recovered log removes part of this effect but does not fully restore the original process structure.

- **Isolated log**: The activity `A_Potential fraud` apears as an isolated branch in all three dependency graphs and Petri nets (disconnected from the main process flow and represented as a separate start-to-end branch in the discovered model). This is caused by the chosen filtering parameters. While the activity itself occurs frequently enough to satisfy min_act_count, its incoming and outgoing relations do not meet the dependency_threshold or min_dfg_occurrences requirements.